# ClinicalBridge — End-to-End Demo

Two complete scenario walkthroughs showing the full multi-agent pipeline:
- **Scenario 1: Missed Medication** — PT-001, hypertension, stopped Lisinopril
- **Scenario 3: Silent Deterioration** — PT-008, heart failure, weight trend

Each run goes Alert → Triage → (EHR + Anamnesis in parallel) → Synthesis → Clinical Context Brief.

In [1]:
import sys
import json
import asyncio
import nest_asyncio
nest_asyncio.apply()
import textwrap
from pathlib import Path
from datetime import datetime

sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

from schemas import RPMAlert
from orchestrator.orchestrator import ClinicalBridgeOrchestrator

SCENARIOS_DIR = Path("../data/scenarios")
VECTORSTORE = "../vectorstore"

/Users/ayhan/Dersler/Prompt/Project/Code/clinicalbridge-capstone-main/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/ayhan/Dersler/Prompt/Project/Code/clinicalbridge-capstone-main/.venv/lib/python3.9/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
def print_brief(brief, scenario_name):
    print(f"{'='*60}")
    print(f"  CLINICAL CONTEXT BRIEF — {scenario_name.upper().replace('_', ' ')}")
    print(f"{'='*60}")
    print(f"Patient:      {brief.patient_id}")
    print(f"Generated:    {brief.generated_at}")
    print(f"Urgency:      {brief.urgency}")
    print(f"Confidence:   {brief.overall_confidence:.0%}")
    print(f"Imm. Review:  {brief.immediate_review_required}")
    print()
    print("ALERT SUMMARY")
    for k, v in brief.alert_summary.items():
        print(f"  {k}: {v}")
    print()
    print("CONTEXTUAL ANALYSIS")
    print(textwrap.fill(brief.contextual_analysis, width=70, initial_indent="  ", subsequent_indent="  "))
    print()
    print("RECOMMENDED ACTIONS")
    for i, action in enumerate(brief.recommended_actions, 1):
        print(f"  {i}. {action.action}")
        print(f"     Rationale: {textwrap.fill(action.rationale, 60, subsequent_indent='               ')}")
        print(f"     Source: {action.evidence_source}  |  Confidence: {action.confidence:.0%}")
    print()
    if brief.uncertainties_and_gaps:
        print("UNCERTAINTIES & GAPS")
        for u in brief.uncertainties_and_gaps:
            print(f"  [{u.type}] {u.description}")
    print(f"{'='*60}")

## Scenario 1: Missed Medication

**Setup:** PT-001 (64M, hypertension + diabetes) stopped Lisinopril 2 weeks ago due to persistent cough. RPM shows rising BP over 3 days, today's reading: 188/112 mmHg.

**Key challenge:** The EHR notes mention the cough discussion (July 2025 visit). The anamnesis has the patient reporting medication non-adherence. The synthesis agent must connect these three data streams to explain why BP is elevated and recommend a specific intervention (switch to ARB) rather than just titrating Lisinopril.

In [3]:
alert_path = SCENARIOS_DIR / "missed_medication" / "input_alert.json"
alert_data = json.loads(alert_path.read_text())

print("Input Alert:")
print(json.dumps(alert_data, indent=2))

Input Alert:
{
  "patient_id": "PT-001",
  "timestamp": "2026-01-29T06:00:00",
  "device_type": "wrist_cuff",
  "measured_values": {
    "systolic_bp": 188.4,
    "diastolic_bp": 112.7,
    "heart_rate": 78.0,
    "spo2": 97.2,
    "weight_kg": 88.6,
    "glucose_mgdl": 118.0
  },
  "alert_category": "URGENT",
  "baseline_thresholds": {
    "systolic_bp": [
      110.0,
      145.0
    ],
    "diastolic_bp": [
      70.0,
      95.0
    ],
    "heart_rate": [
      50.0,
      100.0
    ],
    "spo2": [
      93.0,
      100.0
    ],
    "weight_kg": [
      85.0,
      92.0
    ],
    "glucose_mgdl": [
      80.0,
      180.0
    ]
  }
}


In [4]:
alert_mm = RPMAlert(**alert_data)
orchestrator = ClinicalBridgeOrchestrator(prompt_version="v4")

print("Running pipeline...")
t0 = datetime.now()
brief_mm = asyncio.run(orchestrator.run(alert_mm))
elapsed = (datetime.now() - t0).total_seconds()
print(f"Time-to-brief: {elapsed:.1f}s\n")

print_brief(brief_mm, "missed_medication")

Running pipeline...


Time-to-brief: 13.4s

  CLINICAL CONTEXT BRIEF — MISSED MEDICATION
Patient:      PT-001
Generated:    2026-01-29 06:00:00
Urgency:      URGENT
Confidence:   60%
Imm. Review:  True

ALERT SUMMARY
  device_type: wrist_cuff
  measured_values: {'systolic_bp': 188.4, 'diastolic_bp': 112.7, 'heart_rate': 78.0, 'spo2': 97.2, 'weight_kg': 88.6, 'glucose_mgdl': 118.0}
  baseline_thresholds: {'systolic_bp': [110.0, 145.0], 'diastolic_bp': [70.0, 95.0], 'heart_rate': [50.0, 100.0], 'spo2': [93.0, 100.0], 'weight_kg': [85.0, 92.0], 'glucose_mgdl': [80.0, 180.0]}
  alert_category: URGENT

CONTEXTUAL ANALYSIS
  The RPM alert indicates significantly elevated systolic (188.4 mmHg)
  and diastolic (112.7 mmHg) blood pressure values, both exceeding
  baseline thresholds by 43.4 and 17.7 points, respectively. These
  values suggest heightened cardiovascular risk. Heart rate, SpO2,
  weight, and glucose are within acceptable ranges. However, no
  relevant EHR data was retrieved regarding hypertension hist

### Triage Trace

Shows the intermediate triage decision before the parallel EHR/Anamnesis agents run.

In [5]:
# Run triage agent directly to inspect intermediate reasoning
from agents.triage_agent import TriageAgent

triage_agent = TriageAgent(prompt_version="v4")
triage_result = triage_agent.run(alert_mm)

print("Triage Decision:")
print(f"  Urgency:          {triage_result.urgency}")
print(f"  Clinical question: {triage_result.clinical_question}")
print(f"  EHR query params:  {triage_result.ehr_query_params}")
print(f"  Anamnesis cats:    {triage_result.anamnesis_categories}")
print(f"\nReasoning:")
print(textwrap.fill(triage_result.reasoning, 70, initial_indent="  ", subsequent_indent="  "))

Triage Decision:
  Urgency:          URGENT
  Clinical question: What is the patient hypertension history, current antihypertensive regimen, and recent medication adherence?
  EHR query params:  {'focus': 'hypertension medications labs blood pressure history'}
  Anamnesis cats:    ['medication_adherence', 'recent_symptoms', 'lifestyle_factors']

Reasoning:
  Systolic BP is 43.4 points above the upper threshold of 145.
  Diastolic BP is 17.7 points above the upper threshold of 95. Both
  values are abnormal simultaneously, compounding cardiovascular risk.
  Heart rate, SpO2, weight, and glucose are within acceptable ranges.
  No RPM trend history is available to assess sustained patterns.
  Values do not meet CRITICAL criteria as systolic BP is below 210 and
  diastolic BP is below life-threatening levels. Prompt clinical
  review is warranted.


---

## Scenario 3: Silent Deterioration

**Setup:** PT-008 (71F, heart failure with reduced ejection fraction). Over the past 10 days, RPM shows weight gain of 3.2 kg. Today's alert: weight 82.7 kg (baseline 79.5 kg). The patient reports ankle swelling in her anamnesis diary but hasn't called the clinic.

**Key challenge:** No single data point screams CRITICAL — but the *trend* across three streams (RPM weight, anamnesis ankle swelling, EHR history of prior decompensations) is a classic heart failure decompensation pattern. The synthesis agent must recognize the convergence.

In [6]:
alert_sd_path = SCENARIOS_DIR / "silent_deterioration" / "input_alert.json"
alert_sd_data = json.loads(alert_sd_path.read_text())

print("Input Alert:")
print(json.dumps(alert_sd_data, indent=2))

Input Alert:
{
  "patient_id": "PT-008",
  "timestamp": "2026-01-28T06:00:00",
  "device_type": "smart_scale",
  "measured_values": {
    "systolic_bp": 120.3,
    "diastolic_bp": 74.1,
    "heart_rate": 78.0,
    "spo2": 96.8,
    "weight_kg": 74.4,
    "glucose_mgdl": 99.0
  },
  "alert_category": "URGENT",
  "baseline_thresholds": {
    "systolic_bp": [
      95.0,
      140.0
    ],
    "diastolic_bp": [
      60.0,
      90.0
    ],
    "heart_rate": [
      50.0,
      100.0
    ],
    "spo2": [
      93.0,
      100.0
    ],
    "weight_kg": [
      69.7,
      72.7
    ],
    "glucose_mgdl": [
      70.0,
      140.0
    ]
  }
}


In [7]:
alert_sd = RPMAlert(**alert_sd_data)
orchestrator2 = ClinicalBridgeOrchestrator(prompt_version="v4")

print("Running pipeline...")
t0 = datetime.now()
brief_sd = asyncio.run(orchestrator2.run(alert_sd))
elapsed = (datetime.now() - t0).total_seconds()
print(f"Time-to-brief: {elapsed:.1f}s\n")

print_brief(brief_sd, "silent_deterioration")

Running pipeline...


Time-to-brief: 10.7s

  CLINICAL CONTEXT BRIEF — SILENT DETERIORATION
Patient:      PT-008
Generated:    2026-01-28 06:00:00
Urgency:      ROUTINE
Confidence:   60%
Imm. Review:  False

ALERT SUMMARY
  device_type: smart_scale
  measured_values: {'systolic_bp': 120.3, 'diastolic_bp': 74.1, 'heart_rate': 78.0, 'spo2': 96.8, 'weight_kg': 74.4, 'glucose_mgdl': 99.0}
  alert_category: URGENT
  baseline_thresholds: {'systolic_bp': [95.0, 140.0], 'diastolic_bp': [60.0, 90.0], 'heart_rate': [50.0, 100.0], 'spo2': [93.0, 100.0], 'weight_kg': [69.7, 72.7], 'glucose_mgdl': [70.0, 140.0]}

CONTEXTUAL ANALYSIS
  The RPM alert flagged the patient's weight as 74.4 kg, which is 1.7
  kg above the upper threshold of 72.7 kg. All other measured values,
  including blood pressure, heart rate, SpO2, and glucose, are within
  normal ranges. The triage decision downgraded the urgency to ROUTINE
  due to the isolated nature of the weight deviation and lack of
  supporting RPM trends. EHR data retrieval was 

### RPM Weight Trend — PT-008

In [8]:
import json
data = json.load(open("../data/rpm/PT-008_rpm.json"))
lo, hi = data["baseline_thresholds"]["weight_kg"]
print(f"PT-008 — daily weight (threshold {lo}-{hi} kg):")
for r in data["readings"]:
    w = r["values"]["weight_kg"]
    flag = "  <-- above threshold" if w > hi else ""
    print(r["timestamp"][:10], w, flag)

PT-008 — daily weight (threshold 69.7-72.7 kg):
2026-01-16 71.1 
2026-01-17 71.5 
2026-01-18 71.7 
2026-01-19 72.0 
2026-01-20 72.2 
2026-01-21 72.4 
2026-01-22 72.7 
2026-01-23 72.9   <-- above threshold
2026-01-24 73.1   <-- above threshold
2026-01-25 73.3   <-- above threshold
2026-01-26 73.7   <-- above threshold
2026-01-27 73.9   <-- above threshold
2026-01-28 74.1   <-- above threshold
2026-01-29 74.4   <-- above threshold


---

## Evaluation Summary — Both Scenarios

In [9]:
print("\n=== SCENARIO EVALUATION SUMMARY ===")
print(f"{'Scenario':<25} {'Urgency':<12} {'Confidence':<12} {'Actions':<8} {'Flags':<6}")
print("-" * 65)

for name, brief in [("missed_medication", brief_mm), ("silent_deterioration", brief_sd)]:
    print(
        f"{name:<25}",
        f"{brief.urgency:<12}",
        f"{brief.overall_confidence:<12.0%}",
        f"{len(brief.recommended_actions):<8}",
        f"{len(brief.uncertainties_and_gaps):<6}",
    )

print()
print("Targets: urgency accuracy ≥90%, confidence ≥70%, time-to-brief <30s")


=== SCENARIO EVALUATION SUMMARY ===
Scenario                  Urgency      Confidence   Actions  Flags 
-----------------------------------------------------------------
missed_medication         URGENT       60%          3        4     
silent_deterioration      ROUTINE      60%          3        3     

Targets: urgency accuracy ≥90%, confidence ≥70%, time-to-brief <30s


---

## Full 5-Scenario Evaluation

Run the evaluation harness to get aggregate metrics across all scenarios.

In [10]:
from evaluation.harness import run_all

results = run_all("v4")
print(f"\nFinal pass rate: {sum(r.get('passed', False) for r in results)}/{len(results)}")

Running scenario: missed_medication ... 

ERROR: [Errno 2] No such file or directory: 'data/scenarios/missed_medication/input_alert.json'
Running scenario: false_alarm ... 

ERROR: [Errno 2] No such file or directory: 'data/scenarios/false_alarm/input_alert.json'
Running scenario: silent_deterioration ... 

ERROR: [Errno 2] No such file or directory: 'data/scenarios/silent_deterioration/input_alert.json'
Running scenario: incomplete_record ... 

ERROR: [Errno 2] No such file or directory: 'data/scenarios/incomplete_record/input_alert.json'
Running scenario: conflicting_data ... 

ERROR: [Errno 2] No such file or directory: 'data/scenarios/conflicting_data/input_alert.json'

  EVALUATION SUMMARY — prompt version: v4
  Scenario pass rate     : 0/5  (target ≥4/5)
  Triage accuracy        : 0%  (target ≥90%)
  Hallucination rate     : 0.0%  (target ≤5%)
  Source traceability    : 0%  (target ≥90%)
  Anamnesis completeness : 0%  (target ≥85%)
  Avg time-to-brief      : 0.0s  (target <30s)
  Retrieval precision    : N/A (run ingest first)
  Retrieval recall       : N/A (run ingest first)
Results saved to evaluation/results/run_v4.json

Final pass rate: 0/5
